# Generative AI as a Task Shock
## Replicability, Commodification, and Firm Performance in the Software Industry

**Author:** Hakan Zeki Gülmez — TUM Applied Econometrics  
**Supervisor:** Prof. Dr. Helmut Farbmacher

---
# Section 1 — Research Design

## Research Question

Do software firms (SIC 7370-7379) experience **heterogeneous financial outcomes** after the ChatGPT shock (November 2022), determined by the **task replicability** of their core product and their **switching cost structure**?

## Three Mechanisms

| Mechanism | Condition | Revenue | Margins |
|-----------|-----------|---------|--------|
| **Substitution** | High replicability + low switching costs | Falls | Falls |
| **Commodification** | High replicability + high switching costs | Flat | Falls |
| **Reinforcement** | Low replicability | Rises | Rises |

## Main Regression Equation

$$Y_{it} = \alpha_i + \delta_t + \beta_1(\text{Post}_t \times \text{Replicability}_i) + \beta_2(\text{Post}_t \times \text{SwitchingCost}_i) + \gamma X_{it} + \varepsilon_{it}$$

- $\alpha_i$: Firm fixed effects
- $\delta_t$: Quarter fixed effects
- $\text{Post}_t = 1$ if quarter $\geq$ 2023 Q1
- Standard errors clustered at firm level
- Wild cluster bootstrap for inference

## Treatment Variables (all pre-Nov 2022)

| Variable | Range | Source |
|----------|-------|--------|
| `task_replicability_i` | [0-1] | SBERT semantic similarity |
| `ai_position_i` | {1, 2, 3} | LLM rubric classification |
| `switching_cost_i` | [0-1] | NRR + contract duration |
| `error_cost_i` | [0-6] | Domain classification |

---
# Section 2 — Data Pipeline

## Pipeline Funnel

```
SEC EDGAR JSON          → 7,509 NYSE/Nasdaq firms
  ↓ SIC 7370-7379 filter
EDGAR Browse API        →   530 software firms
  ↓ IPO before 2020
EDGAR Submissions API   →   287 eligible firms
  ↓ B2B focus + manual review
Manual classification   →   155 final sample firms
  ↓ XBRL quarterly data
Financial panel         → 3,034 firm-quarter obs
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

panel = pd.read_csv('../data/raw/financials_panel.csv')
universe = pd.read_csv('../data/raw/firm_universe_v1.csv')
sample = universe[universe['meets_filters'] == True]

print(f'Panel shape:    {panel.shape}')
print(f'Unique firms:   {panel["cik"].nunique()}')
print(f'Date range:     {panel["period_end"].min()} to {panel["period_end"].max()}')
print(f'Fiscal years:   {sorted(panel["fiscal_year"].unique())}')

### Revenue Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log revenue distribution
log_rev = np.log10(panel['revenue'].dropna())
axes[0].hist(log_rev, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('log10(Revenue USD)')
axes[0].set_ylabel('Count')
axes[0].set_title('Revenue Distribution (log scale)')
axes[0].axvline(np.log10(12.5e6), color='red', linestyle='--', label='$12.5M threshold')
axes[0].legend()

# Revenue by year
yearly = panel.groupby('fiscal_year')['revenue'].median() / 1e6
axes[1].bar(yearly.index, yearly.values, color='steelblue', alpha=0.8)
axes[1].set_xlabel('Fiscal Year')
axes[1].set_ylabel('Median Quarterly Revenue ($M)')
axes[1].set_title('Median Revenue by Year')

plt.tight_layout()
plt.savefig('../figures/revenue_distribution.png', bbox_inches='tight')
plt.show()

### Operating Margin Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gross margin
gm = panel['gross_margin'].dropna()
gm_clipped = gm.clip(-1, 1)
axes[0].hist(gm_clipped, bins=50, color='forestgreen', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Gross Margin')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Gross Margin Distribution (n={len(gm):,})')
axes[0].axvline(gm.median(), color='red', linestyle='--', label=f'Median: {gm.median():.2f}')
axes[0].legend()

# Operating margin
om = panel['operating_margin'].dropna()
om_clipped = om.clip(-2, 1)
axes[1].hist(om_clipped, bins=50, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Operating Margin')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Operating Margin Distribution (n={len(om):,})')
axes[1].axvline(om.median(), color='red', linestyle='--', label=f'Median: {om.median():.2f}')
axes[1].axvline(0, color='black', linestyle='-', alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/margin_distributions.png', bbox_inches='tight')
plt.show()

### Coverage Heatmap (Firms x Quarters)

In [ ]:
# Create period label
panel['period'] = panel['fiscal_year'].astype(str) + 'Q' + panel['fiscal_quarter'].astype(str)
coverage = panel.pivot_table(index='ticker', columns='period', values='revenue', aggfunc='count')
coverage = coverage.fillna(0).clip(0, 1)

# Sort columns chronologically
cols = sorted(coverage.columns, key=lambda x: (int(x[:4]), int(x[-1])))
coverage = coverage[cols]

fig, ax = plt.subplots(figsize=(18, 30))
sns.heatmap(coverage, cmap=['#f0f0f0', 'steelblue'], cbar=False,
            linewidths=0.1, linecolor='white', ax=ax)
ax.set_title('Data Coverage: Firms x Quarters (blue = data exists)', fontsize=14)
ax.set_xlabel('Quarter')
ax.set_ylabel('Ticker')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=6)

# Add shock line
if '2023Q1' in cols:
    shock_idx = cols.index('2023Q1')
    ax.axvline(shock_idx, color='red', linewidth=2, label='ChatGPT Shock')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../figures/coverage_heatmap.png', bbox_inches='tight')
plt.show()

---
# Section 3 — Upcoming Steps

| Step | Script | Status |
|------|--------|--------|
| 5 | `collect_product_text.py` — Wayback Machine snapshots (Q2-Q3 2022) | ✅ 106 firms |
| 5b | `apply_10k_fallback.py` — 10-K fallback for missing firms | ✅ 37 firms |
| 6 | `build_replicability.py` — SBERT semantic similarity scores (18 anchors) | ✅ 143/143 |
| 7 | `build_master_panel.py` — Merge financials + replicability | ✅ 2,982 rows |
| 8 | `analysis/did_main.R` — DiD, event study, parallel trends | ✅ Complete |
| 9 | Wild cluster bootstrap inference | Pending |
| 10 | Heterogeneity analysis (AI position, switching costs) | Pending |

---
# Section 4 — Sample Firms

In [ ]:
sample_display = sample[['ticker', 'company_name', 'sic_code', 'exchange']].sort_values('ticker').reset_index(drop=True)
print(f'Total firms in sample: {len(sample_display)}')
print()

# SIC breakdown
sic_names = {
    7370: 'Computer Services',
    7371: 'Computer Programming',
    7372: 'Prepackaged Software',
    7373: 'Computer Integrated Systems',
    7374: 'Computer Processing & Data Prep',
}
print('SIC Code Breakdown:')
for sic, count in sample_display['sic_code'].value_counts().sort_index().items():
    print(f'  {sic} ({sic_names.get(sic, "Other")}): {count}')

print(f'\nExchange Breakdown:')
print(sample_display['exchange'].value_counts().to_string())
print()
sample_display

---
# Section 5 — Main Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- TABLE 2: Main DiD Results ---
print("=" * 70)
print("TABLE 2: MAIN DiD RESULTS")
print("Specification: bare DiD, trimmed 2020+ sample, firm + quarter FE")
print("=" * 70)
print()
print(f"{'Outcome':<20} {'Coeff':>8} {'SE':>10} {'95% CI':>20} {'Sig':>6}")
print("-" * 70)

results = [
    ("ln(Revenue)",      -0.759, 0.401, "*"),
    ("Gross Margin",      0.054, 0.083, ""),
    ("Operating Margin", -0.434, 0.403, ""),
]

for name, coef, se, sig in results:
    ci_lo = coef - 1.96 * se
    ci_hi = coef + 1.96 * se
    print(f"{name:<20} {coef:>8.3f} ({se:.3f})  [{ci_lo:>6.3f}, {ci_hi:>6.3f}] {sig:>5}")

print("-" * 70)
print("Observations: 2,585  |  Firms: 143  |  Period: 2020 Q1 - 2025 Q4")
print("Clustered SE at firm level. * p<0.10, ** p<0.05, *** p<0.01")
print()
print("Parallel Trends (Wald joint F-test, 2020+ sample):")
print("  ln(Revenue):      F=0.89, p=0.54 (PASS)")
print("  Gross Margin:     F=0.98, p=0.46 (PASS)")
print("  Operating Margin: F=1.00, p=0.44 (PASS)")

# --- Coefficient Plot ---
fig, ax = plt.subplots(figsize=(10, 4))

outcomes = ["Operating\nMargin", "Gross\nMargin", "ln(Revenue)"]
coefs = [-0.434, 0.054, -0.759]
ses = [0.403, 0.083, 0.401]
ci_los = [c - 1.96 * s for c, s in zip(coefs, ses)]
ci_his = [c + 1.96 * s for c, s in zip(coefs, ses)]
colors = ["steelblue", "gray", "steelblue"]
y_pos = range(len(outcomes))

ax.barh(y_pos, coefs, color=colors, alpha=0.7, height=0.5, edgecolor="white")
ax.errorbar(coefs, y_pos, xerr=[[c - lo for c, lo in zip(coefs, ci_los)],
                                  [hi - c for c, hi in zip(coefs, ci_his)]],
            fmt="none", ecolor="black", capsize=5, linewidth=1.5)

ax.axvline(0, color="black", linewidth=0.8, linestyle="-")
ax.set_yticks(y_pos)
ax.set_yticklabels(outcomes, fontsize=12)
ax.set_xlabel("Coefficient (Post x Replicability)", fontsize=12)
ax.set_title("Main DiD Coefficients with 95% CI\n(Trimmed 2020+ sample, bare specification)",
             fontsize=13, fontweight="bold")

# Mark significance
ax.annotate("*", xy=(coefs[2], 2), fontsize=18, fontweight="bold",
            color="darkred", ha="left", va="center",
            xytext=(coefs[2] + 0.05, 2))

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("../figures/main_did_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved: figures/main_did_coefficients.png")